# DDSP Baseline: Differentiable Digital Signal Processing

## What is DDSP?

Differentiable Digital Signal Processing (Engel et al., ICLR 2020) fuses classical **vocoders/synthesizers** with modern **deep learning**. The key insight: instead of generating audio samples directly, we train a neural network to predict *synthesis parameters* (pitch, harmonics, noise) for a classical signal processing chain. Because every step is differentiable, gradients flow end-to-end.

### Architecture
```
Audio → [Encoder] → (F0, Loudness) → [Decoder: MLP+GRU] → Synth Params
                                                               ↓
                                         Harmonic Oscillator + Filtered Noise
                                                               ↓ dry signal
                                                    Reverb (learned IR)
                                                               ↓
                                              Multi-Scale Spectral Loss
```

The encoder is **hand-crafted** (no learned weights):
- **Loudness**: A-weighted log power, per frame
- **F0**: fundamental frequency via CREPE (pretrained pitch detector)

The decoder is **learned**: a small MLP→GRU→MLP stack mapping 2 scalars per frame to hundreds of synth parameters.

### Why is DDSP great for Active Divergence?

DDSP gives us a **differentiable, interpretable parameter space**:
- We can set F0 and loudness to anything (even extrapolate outside training data)
- Gradients flow through the synthesizer, so we can optimise inputs w.r.t. any perceptual objective
- The two control axes (pitch, loudness) are *semantically meaningful* and independent

This makes DDSP the ideal substrate for **inference-time active divergence**: instead of training a new model, we manipulate the control signals at inference time to steer the synthesis toward novel or constrained outputs. Subsequent notebooks will exploit exactly this.

## Setup

This notebook requires the `active-divergence` conda environment:

```bash
conda env create -f ../environment.yml
conda activate active-divergence
jupyter notebook
```

All model code lives in `../src/ddsp/`. The notebook only **imports and demonstrates**; no model definitions here.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

SAMPLE_RATE = 16000
HOP_LENGTH = 64
DEVICE = 'cpu'  # Notebook runs on CPU; checkpoint loaded with map_location='cpu'

print('PyTorch:', torch.__version__)
print('Running on:', DEVICE)

## Dataset: Why Violin?

We use **violin stems** from the URMP dataset (Li et al., 2018) — a multi-modal multi-instrument dataset with:
- High-quality 48kHz mono WAV recordings
- Separated per-instrument stems
- Ground-truth F0 annotations per frame

**Instrument choice: Violin.** Reasons:
1. Rich harmonic content — gives the harmonic oscillator a lot to model
2. Wide pitch range (G3–A7 ≈ 196–3136 Hz) — stresses F0 extraction
3. Clear monophonic solo lines — avoids polyphony complications
4. DDSP's original paper also trains on solo violin; provides a direct comparison baseline

**Stems selected** (4 pieces, ~10 minutes total):
- `09_Jesus` (Jesus Bleibet Meine Freude) — lyrical, sustained notes
- `17_Nocturne` (Nocturne) — expressive, dynamic range
- `26_King` (In the Hall of the Mountain King) — rhythmic, varied articulation
- `44_K515` (String Quintet K515) — classical, full violin range

Each stem is resampled to **16 kHz**, sliced into **4-second clips** (64,000 samples), and clips with less than 25% voiced frames are dropped to exclude lead-in silence and rests. This leaves 132 training clips.

In [ ]:
# Play a few sample clips from each piece
sample_files = {
    'Jesus (lyrical)':       '../samples/AuSep_2_vn_09_Jesus_16k.wav',
    'Nocturne (expressive)': '../samples/AuSep_1_vn_17_Nocturne_16k.wav',
    'King (rhythmic)':       '../samples/AuSep_1_vn_26_King_16k.wav',
    'K515 (classical)':      '../samples/AuSep_1_vn_44_K515_16k.wav',
}

for name, path in sample_files.items():
    wav, sr = torchaudio.load(path)
    clip = wav[0, sr * 5 : sr * 13].numpy()  # skip any lead-in silence, play 8s
    print(f'--- {name} ---')
    display(Audio(clip, rate=sr))

## Feature Extraction: Loudness and F0

The DDSP encoder produces two per-frame control signals:

**Loudness** — A-weighted log power spectrum, sampled at 250 Hz (hop = 64 samples at 16 kHz).
A-weighting mimics human hearing sensitivity (boosts 1–5 kHz, cuts bass).

**F0** — Fundamental frequency in Hz, extracted by [CREPE](https://github.com/marl/crepe) (a CNN pretrained on pitch detection). Unvoiced frames (silence, noise) are set to 0 Hz.

In [ ]:
from src.ddsp.features import extract_loudness, extract_f0

# Load one clip for demonstration
wav, sr = torchaudio.load('../samples/AuSep_2_vn_09_Jesus_16k.wav')
audio = wav[0, sr * 5 : sr * 13]  # 8 seconds, past any lead-in

loudness = extract_loudness(audio, sr=SAMPLE_RATE, hop_length=HOP_LENGTH)
f0_hz, voiced = extract_f0(audio, sr=SAMPLE_RATE, hop_length=HOP_LENGTH, device='cpu')

t = np.arange(len(loudness)) * HOP_LENGTH / SAMPLE_RATE

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

ax1.plot(t, loudness.numpy(), color='steelblue')
ax1.set_ylabel('Loudness (normalised dB)')
ax1.set_title('A-weighted Log Loudness')
ax1.grid(True, alpha=0.3)

ax2.plot(t, f0_hz.numpy(), color='darkorange')
ax2.set_ylabel('F0 (Hz)')
ax2.set_xlabel('Time (s)')
ax2.set_title('Fundamental Frequency (CREPE)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Frames: {len(loudness)} | Frame rate: {SAMPLE_RATE/HOP_LENGTH:.0f} Hz')
print(f'F0 range (voiced): {f0_hz[voiced].min():.1f} – {f0_hz[voiced].max():.1f} Hz')
print(f'Voiced fraction: {voiced.float().mean():.1%}')

## Model Architecture

### Decoder: MLP + GRU

The decoder maps `(f0_hz, loudness)` per frame to synth parameters:
- **Input MLP** (3 layers, 512 hidden, LayerNorm + ReLU): projects 2 scalars → 512-dim
- **GRU** (1 layer, 512 hidden): adds temporal context across frames
- **Output MLP** (3 layers): shared feature extraction
- **Output heads**:
  - `global_amp`: (B, N) — overall amplitude envelope
  - `harmonic_dist`: (B, N, 100) — per-harmonic amplitude distribution (softmax → sums to 1)
  - `noise_mag`: (B, N, 65) — noise filter frequency magnitudes

All outputs use **modified sigmoid** (Engel et al., appendix): `2σ(x)^2.3 + ε` — ensures positivity.

### Harmonic Oscillator (§3.2)

$$x(n) = A(n) \sum_{k=1}^{K} c_k(n) \sin\!\left(2\pi \sum_{m=0}^{n} k f_0(m)\right)$$

Amplitudes $A_k(n) = A(n) c_k(n)$ are interpolated from frame rate to audio rate. Harmonics above Nyquist are zeroed for anti-aliasing.

### Filtered Noise (§3.4–3.5)

The decoder predicts per-frame FIR filter magnitudes $H_l$. White noise frames are convolved with the IDFT of $H_l$ (Hann-windowed), then overlap-added.

### Reverb (§3.6)

A single **learned impulse response** of 4 seconds (64,000 samples at 16 kHz) is shared across all training examples. It is applied as a frequency-domain multiplication:

$$y = \text{IRFFT}(\text{RFFT}(x_{dry}) \cdot \text{RFFT}(h))$$

Because all violin recordings share the same room, a single fixed IR can factor out the room acoustics completely — leaving the decoder free to model only the instrument's dry timbre. This is the key architectural decision that separates this implementation from a naive harmonic+noise model.

### Loss: Multi-Scale Spectral (§4.2.1)

$$\mathcal{L} = \sum_i \|S_i - \hat{S}_i\|_1 + \|\log S_i - \log \hat{S}_i\|_1$$

Summed over FFT sizes {2048, 1024, 512, 256, 128, 64} with 75% overlap.

In [ ]:
from src.ddsp import DDSPAutoencoder

model = DDSPAutoencoder(
    n_harmonics=100,
    n_noise_bands=65,
    hidden_size=512,
    n_mlp_layers=3,
    sample_rate=SAMPLE_RATE,
    hop_length=HOP_LENGTH,
    reverb_ir_length=64000,
)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')
print(f'  Decoder:  {sum(p.numel() for p in model.decoder.parameters()):,}')
print(f'  Reverb IR: {model.reverb.ir.numel():,}')

## Load Trained Checkpoint and Reconstruct

In [ ]:
CHECKPOINT = '../models/ddsp_baseline_violin.pt'

ckpt = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print(f"Loaded checkpoint from step {ckpt['step']} (val_loss={ckpt['val_loss']:.4f})")

In [ ]:
wav, sr = torchaudio.load('../samples/AuSep_1_vn_17_Nocturne_16k.wav')
start = sr * 30
audio_orig = wav[0, start : start + sr * 4]

with torch.no_grad():
    loudness_in = extract_loudness(audio_orig, sr=SAMPLE_RATE, hop_length=HOP_LENGTH)
    f0_in, _ = extract_f0(audio_orig, sr=SAMPLE_RATE, hop_length=HOP_LENGTH, device='cpu')

    N = min(loudness_in.shape[0], f0_in.shape[0])
    out = model(f0_in[:N].unsqueeze(0), loudness_in[:N].unsqueeze(0))
    audio_recon = out['audio'][0]

T = min(audio_orig.shape[0], audio_recon.shape[0])
audio_orig  = audio_orig[:T]
audio_recon = audio_recon[:T]

print('Original:')
display(Audio(audio_orig.numpy(), rate=SAMPLE_RATE))
print('Reconstructed (with learned reverb):')
display(Audio(audio_recon.detach().numpy(), rate=SAMPLE_RATE))

In [ ]:
# Spectrogram comparison
def spectrogram(audio, n_fft=1024, hop=256):
    S = torch.stft(
        audio, n_fft=n_fft, hop_length=hop,
        win_length=n_fft,
        window=torch.hann_window(n_fft),
        return_complex=True
    ).abs()
    return 20 * torch.log10(S.clamp(min=1e-5)).numpy()

S_orig  = spectrogram(audio_orig)
S_recon = spectrogram(audio_recon.detach())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
vmin, vmax = -80, 0

ax1.imshow(S_orig, aspect='auto', origin='lower', vmin=vmin, vmax=vmax,
           extent=[0, T/SAMPLE_RATE, 0, SAMPLE_RATE/2/1000])
ax1.set_title('Original')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Freq (kHz)')

ax2.imshow(S_recon, aspect='auto', origin='lower', vmin=vmin, vmax=vmax,
           extent=[0, T/SAMPLE_RATE, 0, SAMPLE_RATE/2/1000])
ax2.set_title('Reconstructed')
ax2.set_xlabel('Time (s)')

plt.suptitle('Spectrogram Comparison (Nocturne, 4s)')
plt.tight_layout()
plt.show()

## Interactive Synthesis: Control F0 and Loudness

The real power of DDSP: we can **synthesize audio from scratch** by providing arbitrary F0 and loudness trajectories — no audio encoder needed at inference time.

The sliders below set a constant pitch (MIDI note) and loudness. Note that for the interactive widget we **bypass the reverb** (`out['dry_audio']`) so the sound is more immediate and responsive to control changes — the reverb IR is a fixed room characteristic, not a creative parameter. Notebook 2 will show how to selectively engage or disengage the reverb as part of active divergence.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

def midi_to_hz(midi: float) -> float:
    return 440.0 * 2.0 ** ((midi - 69) / 12.0)

def synthesise_from_controls(midi_note: float, loudness_val: float, duration_s: float, use_reverb: bool = False):
    n_frames = int(duration_s * SAMPLE_RATE / HOP_LENGTH)
    f0   = torch.full((1, n_frames), midi_to_hz(midi_note))
    loud = torch.full((1, n_frames), loudness_val)

    with torch.no_grad():
        out = model(f0, loud)
    audio = out['audio' if use_reverb else 'dry_audio'][0].numpy()
    peak = np.abs(audio).max()
    if peak > 0:
        audio = audio / peak * 0.8
    return audio

midi_slider   = widgets.FloatSlider(value=69, min=40, max=88, step=1,
                                    description='MIDI pitch', continuous_update=False)
loud_slider   = widgets.FloatSlider(value=0.5, min=-0.5, max=1.5, step=0.05,
                                    description='Loudness', continuous_update=False)
dur_slider    = widgets.FloatSlider(value=2.0, min=0.5, max=4.0, step=0.5,
                                    description='Duration (s)', continuous_update=False)
reverb_toggle = widgets.Checkbox(value=False, description='Apply reverb')
output_widget = widgets.Output()

def on_change(_):
    audio = synthesise_from_controls(
        midi_slider.value, loud_slider.value, dur_slider.value, reverb_toggle.value
    )
    hz = midi_to_hz(midi_slider.value)
    with output_widget:
        clear_output(wait=True)
        print(f'F0={hz:.1f} Hz (MIDI {midi_slider.value:.0f}), '
              f'Loudness={loud_slider.value:.2f}, Duration={dur_slider.value:.1f}s')
        display(Audio(audio, rate=SAMPLE_RATE, autoplay=False))

for w in (midi_slider, loud_slider, dur_slider, reverb_toggle):
    w.observe(on_change, names='value')

display(widgets.VBox([midi_slider, loud_slider, dur_slider, reverb_toggle, output_widget]))
on_change(None)

## What's Next?

This notebook established a baseline DDSP model trained to reconstruct solo violin audio from F0 and loudness alone.

**Notebook 02 — Inference-Time Active Divergence** will exploit this differentiable synthesizer:

1. **Gradient-based control optimisation**: Start from an arbitrary F0/loudness trajectory and use gradients through the DDSP synthesizer to steer audio toward a perceptual target (e.g., a spectral centroid target, a pitch profile).

2. **Latent space manipulation**: Use the interpretable control axes to interpolate, extrapolate, and morph between sound identities.

3. **Constrained synthesis**: Apply hard constraints (e.g., "must be a C major scale") while optimising softly toward an aesthetic target.

The key idea: **active divergence** means deviating from the training distribution *intentionally* and *controllably* — exactly what DDSP's interpretable parameter space enables.